# 🚀 Complete Production Pipeline: Prefect + dbt + Snowflake

---

## 🏗️ What We're Building

```
production_snowflake_pipeline(run_date, target, full_refresh)
    │
    ├── Task 1: Load Snowflake Credentials  (from Prefect Secrets)
    ├── Task 2: dbt seed                    → CSV → Snowflake RAW schema
    ├── Task 3: dbt run                     → Staging → Marts in Snowflake
    ├── Task 4: dbt test                    → Validate Snowflake data quality
    ├── Task 5: Pipeline Report             → Summary with durations
    ├── on_failure  → Alert on crash
    └── on_success  → Alert on completion
```

---

## 💻 Example 1: Secure Credential Loading from Prefect Secrets

In [ ]:
from prefect import flow, task
from prefect.blocks.system import Secret
from datetime import date
import subprocess, os, time

DBT_PROJECT_DIR = "/Users/aviraljain/Downloads/python advanced/my_etl_project"

# ─── Credential Management ────────────────────────────────────────────────────
def get_snowflake_env() -> dict:
    """
    Load Snowflake credentials securely:
    1. Try Prefect Secret blocks first (recommended for production)
    2. Fall back to environment variables (useful for local dev)
    """
    env = os.environ.copy()
    secret_map = {
        "SNOWFLAKE_ACCOUNT":   "snowflake-account",
        "SNOWFLAKE_USER":      "snowflake-user",
        "SNOWFLAKE_PASSWORD":  "snowflake-password",
        "SNOWFLAKE_WAREHOUSE": "snowflake-warehouse",
        "SNOWFLAKE_DATABASE":  "snowflake-database",
    }
    loaded = []
    for env_key, block_name in secret_map.items():
        try:
            env[env_key] = Secret.load(block_name).get()
            loaded.append(env_key)
        except Exception:
            pass  # Falls back to os.environ
    if loaded:
        print(f"🔐 Loaded from Prefect Secrets: {loaded}")
    return env


# ─── dbt Runner ───────────────────────────────────────────────────────────────
def run_dbt(command: str, target: str = "dev", extra_flags: list = None) -> str:
    cmd = ["dbt"] + command.split() + [
        "--target", target,
        "--project-dir", DBT_PROJECT_DIR,
        "--profiles-dir", DBT_PROJECT_DIR
    ]
    if extra_flags: cmd += extra_flags
    result = subprocess.run(cmd, capture_output=True, text=True, env=get_snowflake_env())
    print(result.stdout[-600:])
    if result.returncode != 0:
        raise RuntimeError(f"dbt failed:\n{result.stderr[-400:]}")
    return result.stdout

print("Setup complete ✅")

---

## 💻 Example 2: Production-Grade Snowflake Tasks

In [ ]:
@task(name="1. dbt seed → Snowflake RAW", retries=1, retry_delay_seconds=30)
def dbt_seed(target: str) -> dict:
    """Upload seed CSVs to Snowflake RAW schema."""
    start = time.time()
    run_dbt("seed", target)
    duration = round(time.time() - start, 2)
    print(f"✅ Seeds loaded into Snowflake ({target}) in {duration}s")
    return {"step": "seed", "duration_sec": duration, "target": target, "status": "success"}


@task(name="2. dbt run → Snowflake models", retries=2, retry_delay_seconds=30)
def dbt_run(target: str, full_refresh: bool = False) -> dict:
    """Run all dbt models. Creates staging/marts in Snowflake."""
    start = time.time()
    extra = ["--full-refresh"] if full_refresh else []
    run_dbt("run", target, extra)
    duration = round(time.time() - start, 2)
    print(f"✅ dbt run complete on Snowflake ({target}) in {duration}s")
    return {"step": "run", "duration_sec": duration, "full_refresh": full_refresh, "status": "success"}


@task(name="3. dbt test → Snowflake", retries=0)
def dbt_test(target: str) -> dict:
    """Run dbt data quality tests on Snowflake. Returns status dict."""
    start = time.time()
    cmd = ["dbt", "test", "--target", target,
           "--project-dir", DBT_PROJECT_DIR,
           "--profiles-dir", DBT_PROJECT_DIR]
    result = subprocess.run(cmd, capture_output=True, text=True, env=get_snowflake_env())
    print(result.stdout[-600:])
    duration = round(time.time() - start, 2)
    passed = result.returncode == 0
    status = "passed" if passed else "failed"
    if not passed:
        print(f"⚠️ Snowflake data quality tests FAILED in {duration}s")
    return {"step": "test", "duration_sec": duration, "status": status, "target": target}


@task(name="4. Pipeline Report")
def generate_report(seed: dict, run: dict, test: dict, run_date: str, target: str):
    """Print a full summary of the Snowflake pipeline run."""
    total = seed['duration_sec'] + run['duration_sec'] + test['duration_sec']
    print("\n" + "═" * 55)
    print("📋  SNOWFLAKE + dbt PIPELINE REPORT")
    print("═" * 55)
    print(f"📅  Run Date      : {run_date}")
    print(f"❄️   Snowflake     : target={target}")
    print(f"🌱  dbt seed      : {seed['status']} ({seed['duration_sec']}s)")
    print(f"▶   dbt run       : {run['status']} (full_refresh={run['full_refresh']}, {run['duration_sec']}s)")
    print(f"🧪  dbt test      : {test['status']} ({test['duration_sec']}s)")
    print(f"⏱   Total Time    : {total}s")
    print("═" * 55)

print("Production tasks defined ✅")

---

## 💻 Example 3: The Complete Production Flow

In [ ]:
# ─── Notification Hooks ───────────────────────────────────────────────────────
def on_failure(flow, flow_run, state):
    print(f"🔴 SNOWFLAKE PIPELINE FAILED: {flow.name}")
    print(f"   State: {state.name} | {state.message}")
    # Add: Slack webhook, PagerDuty, email, etc.

def on_success(flow, flow_run, state):
    print(f"🟢 SNOWFLAKE PIPELINE SUCCEEDED: {flow.name}")


# ─── Production Flow ──────────────────────────────────────────────────────────
@flow(
    name="Production: Prefect + dbt + Snowflake",
    log_prints=True,
    on_failure=[on_failure],
    on_completion=[on_success]
)
def production_snowflake_pipeline(
    run_date: str   = str(date.today()),
    target: str     = "dev",
    full_refresh: bool = False
):
    """
    Production-grade Prefect + dbt + Snowflake pipeline.

    Parameters:
        run_date     : Date this pipeline is processing (default: today)
        target       : Snowflake target — 'dev' or 'prod'
        full_refresh : Rebuild all incremental Snowflake tables if True
    """
    print(f"🚀 Starting Snowflake pipeline | date={run_date} target={target} full_refresh={full_refresh}")

    seed_meta = dbt_seed(target)
    run_meta  = dbt_run(target, full_refresh)
    test_meta = dbt_test(target)
    generate_report(seed_meta, run_meta, test_meta, run_date, target)

    if test_meta["status"] == "failed":
        print("\n⚠️ Pipeline done — but Snowflake data quality issues detected!")
    else:
        print("\n✅ Snowflake pipeline 100% complete — all tests passed!")


# 🚀 Run it!
production_snowflake_pipeline(
    run_date=str(date.today()),
    target="dev",
    full_refresh=False
)

---

## 💻 Example 4: Schedule This for Daily Production Runs

In [ ]:
schedule_example = '''
# Save as: snowflake_prod_pipeline.py
# Then run: python snowflake_prod_pipeline.py

if __name__ == "__main__":
    production_snowflake_pipeline.serve(
        name="prod-snowflake-daily",
        cron="0 6 * * *",               # 6 AM every day 
        timezone="Asia/Kolkata",         # IST timezone
        parameters={
            "target": "prod",
            "full_refresh": False        # Full refresh only manually
        }
    )
'''
print("Schedule this pipeline:")
print(schedule_example)

# To trigger a manual backfill from CLI:
print("Run a manual backfill from CLI:")
print("  prefect deployment run 'Production: Prefect + dbt + Snowflake/prod-snowflake-daily'")
print("  ... with params:")
print("  --param run_date=2024-01-01 --param full_refresh=true")

---

## 🏭 Full Architecture Recap

```
production_snowflake_pipeline(run_date, target, full_refresh)
    │
    ├── get_snowflake_env()           → Load credentials from Prefect Secrets
    │
    ├── dbt_seed(target)              → CSV → Snowflake RAW schema
    │       retries=1, delay=30s
    │
    ├── dbt_run(target, full_refresh) → Staging/Marts tables in Snowflake
    │       retries=2, delay=30s
    │
    ├── dbt_test(target)              → Data quality on Snowflake (returns dict)
    │       no retry
    │
    ├── generate_report(...)          → Print run summary
    │
    ├── on_failure → Alert
    └── on_success → Confirm
```

---

## ⚠️ Common Beginners' Mistakes

In [ ]:
mistakes = [
    ("Hardcoded Snowflake password",    "Use Prefect Secret blocks — rotate without changing code"),
    ("target='dev' in production",     "Always pass target='prod' to the deployed schedule"),
    ("full_refresh daily",             "Very expensive - full_refresh rebuilds all Snowflake tables"),
    ("No report/metadata from tasks",  "Return dicts from tasks for audit trails and debugging"),
    ("No on_failure hook",             "You WILL miss failures without it — Snowflake may process bad data"),
]
for mistake, fix in mistakes:
    print(f"❌ {mistake}")
    print(f"✅ Fix: {fix}\n")